# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ErenSnowh/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

- **Unit of analysis:** One row = one content page (pseudonymized `content_hash_id`) for a specific client (`client_hash_id`).
- **Tables used:** `fact_content_daily_performance` (aggregated to the monthly grain) joined with `dim_content` and `dim_clients`.
- **Time window:** The month of March 2026 (`month=2026-03`), which acts as our mid-panel iteration slice.
- **Label / proxy:** `is_declining_label` (predicting whether a page's impressions will drop by >20% based on the next window).
- **Deliberately excluded:** We deliberately exclude `health_score` because it represents a downstream product decision, and we exclude `trend_direction` and `trend_pct` because they perfectly encode the label.

In [1]:
import duckdb
import os
from dotenv import load_dotenv

# Load .env from repo root if running locally
load_dotenv('../../.env')
load_dotenv('.env')

# Connect and set up the Hugging Face token secret
token = os.environ.get('HF_TOKEN')
if not token:
    raise ValueError("HF_TOKEN environment variable not set. Please set it to your Hugging Face read token.")

con = duckdb.connect()
con.execute("INSTALL httpfs;")
con.execute("LOAD httpfs;")
con.execute(f"""
    CREATE SECRET hf (
        TYPE HUGGINGFACE,
        TOKEN '{token}'
    );
""")

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Features:**
  - `impressions_90d`: Knowable at the decision moment because it aggregates historical Google Search Console data.
  - `clicks_90d`: Knowable at the decision moment because it counts past clicks up to the snapshot.
  - `avg_position`: Knowable at the decision moment because it's the average rank over the trailing window.
  - `content_age_days`: Knowable at the decision moment from the content metadata publication date.
  - `word_count`: Knowable at the decision moment directly from the page text.
- **Label:** `is_declining_label` (the target we are predicting).
- **Context:** `content_hash_id`, `client_hash_id`, `report_date` (used only for joins, grouping, and train/test splitting).
- **Excluded:** 
  - `trend_pct`: It is the exact numeric input used to define the `is_declining_label` threshold.
  - `health_score`: It is a product-generated flag that summarizes the page's state, leading to circular logic if learned by the model.

In [2]:
# Define the base path for our mid-panel month partition
fact_table_path = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/**/*.parquet"

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Here we prove the grain (no duplicates when grouped by client, content, and date), show the row count and date span for the `2026-03` slice, and demonstrate data availability logic using `IS TRUE` for the `ga4_data_available` flag.

In [3]:
# Query 1: Prove the grain (one row per client, content, report_date)
grain_check = con.execute(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as c
    FROM read_parquet('{fact_table_path}')
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING c > 1
    LIMIT 5
""").df()
print("Grain violations (should be empty):\n", grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain violations (should be empty):
 Empty DataFrame
Columns: [client_hash_id, content_hash_id, report_date, c]
Index: []


In [4]:
# Query 2: Row count and date span for the mid-panel month
span_check = con.execute(f"""
    SELECT 
        COUNT(*) as total_rows,
        MIN(report_date) as start_date,
        MAX(report_date) as end_date
    FROM read_parquet('{fact_table_path}')
""").df()
print("\nSlice row count and date span:\n", span_check)


Slice row count and date span:
    total_rows start_date   end_date
0     9841378 2026-03-01 2026-03-31


In [5]:
# Query 3: Availability — filter with IS TRUE to safely handle three-valued logic (True, False, NULL)
availability_check = con.execute(f"""
    SELECT 
        COUNT(*) as total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as rows_with_ga4_true,
        SUM(CASE WHEN ga4_data_available IS NOT TRUE THEN 1 ELSE 0 END) as rows_with_ga4_false_or_null
    FROM read_parquet('{fact_table_path}')
""").df()
print("\nAvailability check (GA4 data):\n", availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Availability check (GA4 data):
    total_rows  rows_with_ga4_true  rows_with_ga4_false_or_null
0     9841378            413966.0                    9427412.0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**The Trap (Leakage Demonstration)**: The target `is_declining_label` is derived from `trend_pct` (impressions drop >20%). If we include `trend_pct` (or `trend_direction`) as a feature, the model learns a simple deterministic rule rather than real world patterns. This is data leakage from the future window. Below we simulate this trap by predicting the label on a local sample.

In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load local sample for the experiment
try:
    df_sample = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
except:
    df_sample = pd.read_csv('data/raw/content_refresh_anonymized.csv')

df_sample['is_declining_label'] = (df_sample['trend_direction'] == 'down').astype(int)

# Select a few honest features + the trap
features_honest = ['impressions_90d', 'clicks_90d', 'avg_position']
# Fill missing trend_pct with 0 so the model can run
df_sample['trend_pct'] = df_sample['trend_pct'].fillna(0)

# Prepare data
X_trap = df_sample[features_honest + ['trend_pct']]
X_honest = df_sample[features_honest]
y = df_sample['is_declining_label']

# Train/Test split
X_train_t, X_test_t, y_train, y_test = train_test_split(X_trap, y, test_size=0.2, random_state=42)
X_train_h, X_test_h, _, _ = train_test_split(X_honest, y, test_size=0.2, random_state=42)

# Train with the trap (Leakage)
clf_trap = RandomForestClassifier(n_estimators=20, max_depth=5, random_state=42)
clf_trap.fit(X_train_t, y_train)
trap_preds = clf_trap.predict(X_test_t)
print(f"Accuracy WITH the trap (trend_pct included): {accuracy_score(y_test, trap_preds):.1%}")

# Train honestly
clf_honest = RandomForestClassifier(n_estimators=20, max_depth=5, random_state=42)
clf_honest.fit(X_train_h, y_train)
honest_preds = clf_honest.predict(X_test_h)
print(f"Accuracy WITHOUT the trap (honest features only): {accuracy_score(y_test, honest_preds):.1%}")
print("\nLesson: The trap yields an artificially near-perfect score because the label is derived directly from trend_pct. We must deliberately exclude it.")

Accuracy WITH the trap (trend_pct included): 100.0%


Accuracy WITHOUT the trap (honest features only): 64.5%

Lesson: The trap yields an artificially near-perfect score because the label is derived directly from trend_pct. We must deliberately exclude it.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.